APPEND lh_vv_gold.enr.gld_enr_poi_info từ lh_vv_silver.dbo.slv_poi_entity_non_enrich.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# [SỬA 21/10] 20261021-refresh-gld-enr-poi-info: nguồn slv_poi_entity_non_enrich (D-01). Bỏ slv_3p_poi, address, contact.
# BEGIN_INFO_PURE
# Mục 2 mapping_gld_enr_poi_info.md: (time_to_spend, best_time, cost_vnd). Khớp đúng chữ; sector lạ hoặc NULL -> other.
SECTOR_LABELS = {
    "accommodation": ("Thời gian lưu trú lý tưởng", "Mùa lưu trú đẹp nhất", "Giá phòng tham khảo"),
    "f_n_b": ("Thời gian dùng bữa", "Thời điểm nên ghé", "Mức giá tham khảo"),
    "attraction": ("Thời gian tham quan", "Mùa đẹp nhất", "Chi phí tham quan"),
    "tour_n_activities": ("Thời lượng hoạt động", "Thời điểm phù hợp", "Chi phí tham khảo"),
    "shopping_n_retail": ("Thời gian mua sắm", "Thời điểm nên ghé", "Chi tiêu tham khảo"),
    "transport_route_base": ("Thời gian đưa đón", "Thời điểm phù hợp", "Chi phí đưa đón"),
    "transport_on_demand": ("Thời gian thuê phổ biến", "Thời điểm nên thuê", "Giá thuê tham khảo"),
    "vehicle_rental": ("Thời gian thuê phổ biến", "Thời điểm nên thuê", "Giá thuê tham khảo"),
    "spa_n_wellness": ("Thời lượng liệu trình", "Thời điểm phù hợp", "Giá dịch vụ tham khảo"),
    "event": ("Thời lượng chương trình", "Thời điểm nên xem", "Giá vé tham khảo"),
}
SECTOR_OTHER = ("Thời lượng trải nghiệm", "Thời điểm phù hợp", "Chi phí tham khảo")
# Mục 1.2. Seed ghi "Month 11–4" (gạch dài), mapping ghi "Month 11-3": nhận cả hai, tháng 1–12.
BEST_TIME_RE = r"^Month ([1-9]|1[0-2])[-–]([1-9]|1[0-2])$"
GOOGLE, PARTNER = "google", "partner"
REQUIRED_COLS = {
    "entity": ("poi_id", "source_name", "source_id", "language_code", "website_url", "ward", "city",
               "avg_range_min", "canonical_candidate"),
    "amenity": ("poi_id", "service_flags"),
    "price": ("poi_id", "price_level", "price_min", "price_max"),
    "i18n": ("business_service_id", "locale", "amentities", "deleted"),
}
# END_INFO_PURE


def _keys(frame, column):
    return frame.select(F.col(column).alias("poi_id")).where(F.col("poi_id").isNotNull()).distinct()


def _label(sector, k):
    # RULE-003. k: 0 time_to_spend, 1 best_time, 2 cost_vnd.
    expr = None
    for name, labels in SECTOR_LABELS.items():
        cond = sector == F.lit(name)
        expr = F.when(cond, F.lit(labels[k])) if expr is None else expr.when(cond, F.lit(labels[k]))
    return expr.otherwise(F.lit(SECTOR_OTHER[k]))


def _time_to_spend_text(minutes, sector):
    # D-16, mục 1.1: xét đúng thứ tự nhánh.
    hours = F.floor(minutes / 60).cast("int")
    rest = minutes % 60
    return (F.when(minutes.isNull() | (minutes <= 0), F.lit("Không cố định"))
             .when(sector == F.lit("accommodation"), F.concat(F.ceil(minutes / 1440).cast("string"), F.lit(" ngày")))
             .when((sector == F.lit("tour_n_activities")) & (minutes > 420), F.lit("Cả ngày"))
             .when(minutes < 60, F.concat(minutes.cast("string"), F.lit(" phút")))
             .when(minutes == 60, F.lit("1 tiếng"))
             .when(rest == 0, F.concat(hours.cast("string"), F.lit(" tiếng")))
             .otherwise(F.concat(hours.cast("string"), F.lit(" tiếng "), rest.cast("string"), F.lit(" phút"))))


def _best_time_text(candidate):
    # D-05, mục 1.2: From-To theo thứ tự trong chuỗi; còn lại Quanh năm.
    value = F.trim(candidate)
    return (F.when(value.rlike(BEST_TIME_RE),
                   F.concat(F.lit("Tháng "), F.regexp_extract(value, BEST_TIME_RE, 1),
                            F.lit(" - Tháng "), F.regexp_extract(value, BEST_TIME_RE, 2)))
             .otherwise(F.lit("Quanh năm")))


def _cost(price_min, price_max):
    # COL-007. D-08, D-10, mục 1.9.3 mapping_place_type.md: đủ hai giá mới tính. Không gọi LLM.
    both = price_min.isNotNull() & price_max.isNotNull()
    return F.when(both, F.round((price_min + price_max) / F.lit(2), 0)).cast("bigint")


def _cost_vnd(cost, sector):
    # D-11, D-15, mục 1.3: làm tròn half-up về nghìn, "Từ 546,000đ". cost NULL -> chuỗi JSON null (D-09, D-10).
    rounded = (F.round(cost.cast("decimal(20,0)") / F.lit(1000), 0) * F.lit(1000)).cast("bigint")
    text = F.concat(F.lit("Từ "), F.format_number(rounded, 0), F.lit("đ"))
    return F.when(cost.isNull(), F.lit("null")).otherwise(F.to_json(F.create_map(_label(sector, 2), text)))


def _review_json(reviews):
    # COL-048 — MQ-053, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P. RULE-006.
    window = Window.partitionBy("poi_id").orderBy(F.col("time").desc(), F.col("review_id").asc())
    top = (reviews.withColumn("rn", F.row_number().over(window)).where(F.col("rn") <= 20)
           .select("poi_id", F.struct(
               F.col("rn"), F.col("review_id"), F.substring(F.col("text"), 1, 350).alias("review_text"),
               F.col("time"), F.col("published_at")).alias("elem")))
    packed = top.groupBy("poi_id").agg(F.collect_list("elem").alias("elems"))
    payload = F.to_json(F.transform(F.array_sort(F.col("elems")), lambda row: F.struct(
        row.review_id.alias("review_id"), row.review_text.alias("review_text"),
        row.time.alias("time"), row.published_at.alias("published_at"))))
    return packed.select("poi_id", payload.alias("review_json"))



def _service_flag(amenity, keys):
    # D-12: một dòng / poi_id: mảng JSON giá trị khác nhau, bỏ NULL, sort tăng dần. Không có dòng -> [] ở build.
    return (amenity.join(keys, "poi_id", "left_semi").groupBy("poi_id")
            .agg(F.to_json(F.array_sort(F.collect_set("service_flags"))).alias("service_flag")))


def _vi_i18n(frame):
    # D-17: chỉ locale = vi. i18n không có poi_id: poi_id partner = uuid_md5(business_service_id, 'partner'),
    # cùng hàm và thứ tự đối số với _partner của NB_SLV_POI_ENTITY_NON_ENRICH.
    return (frame.where(F.col("locale") == F.lit("vi"))
            .withColumn("poi_id", uuid_md5(F.col("business_service_id"), F.lit(PARTNER))))


def _partner_hours(ctx):
    # D-17: bỏ dòng deleted = true (như _live của N6). Không có key policies.operating_hours -> NULL.
    live = ctx.input("i18n").where(~F.coalesce(F.col("deleted"), F.lit(False)))
    return _vi_i18n(live).select(
        "poi_id", F.get_json_object(F.col("amentities"), "$.policies.operating_hours").alias("partner_hours"))


def _i18n_keys(ctx):
    # D-17: bản i18n vi đổi, kể cả deleted = true -> poi_id partner. build lọc lại theo spine.
    return _keys(_vi_i18n(ctx.changed("i18n")), "poi_id")


def _role_keys(ctx, role, spine_keys):
    # SPARK-001: ctx.changed là current.subtract(old), nên dòng biến mất (xoá mềm) không nằm trong đó.
    # rating, price, hours, review, amenity: thêm poi_id của dòng có ở baseline mà không còn ở version đọc.
    # Không có baseline -> mọi POI. Gọi changed trước vì nó có thể đổi basis sang FULL_VERSION_UNAVAILABLE.
    forward = _keys(ctx.changed(role), "poi_id")
    info = ctx.basis[role]
    if info["basis"] not in ("EDGE", "MARKER") or info["from"] is None:
        return spine_keys
    if info["from"] >= info["to"]:
        return forward
    current = ctx._input_at(role, info["to"])
    old = _align_snapshot(current, ctx._input_at(role, info["from"]))
    return forward.union(_keys(old.subtract(current), "poi_id"))


def _require(ctx):
    for role, cols in REQUIRED_COLS.items():
        have = {c.lower() for c in ctx.input(role).columns}
        missing = [c for c in cols if c not in have]
        if missing:
            raise ConfigError(f"gld_enr_poi_info: {ctx.spec.inputs[role]} thiếu cột {missing}")


def build(ctx: NodeContext) -> DataFrame:
    _require(ctx)
    spine = ctx.input("entity")  # D-04: lib đã bỏ dòng xoá mềm và row_hash, created_at, updated_at, deleted_at
    spine_keys = spine.select("poi_id")
    changed = _keys(ctx.changed("entity"), "poi_id")
    for role in ("rating", "price", "hours", "review", "amenity"):
        changed = changed.union(_role_keys(ctx, role, spine_keys))
    changed = (changed.union(_i18n_keys(ctx))
               .distinct().join(spine_keys, "poi_id", "left_semi")).persist(StorageLevel.MEMORY_AND_DISK)
    ctx.cached.append(changed)
    dropped = _keys(ctx.changed("review"), "poi_id").join(spine_keys, "poi_id", "left_anti").count()
    ctx.warn("review_poi_not_in_entity", dropped)
    e = spine.join(changed, "poi_id", "left_semi").alias("e")
    am = _service_flag(ctx.input("amenity"), changed).alias("am")
    r = ctx.input("rating").alias("r")
    pr = ctx.input("price").alias("pr")
    oh = ctx.input("hours").alias("oh")
    rj = _review_json(ctx.input("review").join(changed, "poi_id", "left_semi")).alias("rj")
    ph = _partner_hours(ctx).join(changed, "poi_id", "left_semi").alias("ph")
    base = (e.join(am, F.col("e.poi_id") == F.col("am.poi_id"), "left")
             .join(r, F.col("e.poi_id") == F.col("r.poi_id"), "left")
             .join(pr, F.col("e.poi_id") == F.col("pr.poi_id"), "left")
             .join(oh, F.col("e.poi_id") == F.col("oh.poi_id"), "left")
             .join(rj, F.col("e.poi_id") == F.col("rj.poi_id"), "left")
             .join(ph, F.col("e.poi_id") == F.col("ph.poi_id"), "left"))
    source = F.lower(F.trim(F.col("e.source_name")))
    google = source == F.lit(GOOGLE)
    sector = F.col("e.business_sector")
    cost = _cost(F.col("pr.price_min"), F.col("pr.price_max"))
    # COL-009 — created_at do lib gán batch_ts. COL-010 — updated_at do lib gán batch_ts.
    return base.select(
        # BEGIN_INFO_SELECT
        F.col("e.poi_id").alias("poi_id"),  # COL-001 — D-01
        F.col("e.source_name").alias("source_name"),
        F.col("e.source_id").alias("source_id"),
        F.col("e.language_code").alias("language_code"),
        F.col("e.poi_name").alias("poi_name"),  # COL-035
        F.col("e.poi_name_normalized").alias("poi_name_normalized"),
        F.col("e.types_json").alias("types"),  # COL-036 — D-13
        F.col("e.operating_status").alias("operating_status"),
        F.col("e.primary_type").alias("primary_type"),
        F.col("e.website_url").alias("website_url"),  # COL-039 — D-04
        F.col("e.house_number").alias("house_number"),
        F.col("e.street").alias("street"),
        F.col("e.ward").alias("ward"),
        F.col("e.ward").alias("district"),  # COL-041 — D-06
        F.col("e.city").alias("city"),
        F.col("e.city").alias("destination_name"),  # COL-042 — D-06
        F.col("e.full_address").alias("full_address"),
        F.col("e.lat").alias("lat"),
        F.col("e.lng").alias("lng"),
        F.col("e.business_sector").alias("business_sector"),  # COL-037
        F.col("e.business_category").alias("business_category"),  # COL-038
        F.col("e.business_category_vi").alias("business_category_vi"),
        F.col("e.category_group").alias("category_group"),
        F.col("e.range_low_min").alias("range_low_min"),
        F.col("e.range_high_min").alias("range_high_min"),
        F.col("e.avg_range_min").alias("avg_range_min"),
        F.col("e.candidate_best_time_code").alias("candidate_best_time_code"),
        F.col("e.canonical_candidate").alias("canonical_candidate"),
        F.when(google, F.col("r.rating_overall")).alias("rating"),  # COL-044 — D-07, D-17
        F.when(google, F.col("r.rating_count")).alias("user_rating_count"),  # COL-045 — D-07
        F.when(google, F.col("pr.price_level")).alias("price_level"),  # COL-046 — D-07
        F.when(google, F.col("oh.periods_json"))
         .when(source == F.lit(PARTNER), F.col("ph.partner_hours")).alias("opening_hours_text"),  # COL-047 — D-17
        F.when(google, F.coalesce(F.col("rj.review_json"), F.lit("[]"))).alias("review_json"),  # COL-048 — D-07
        F.to_json(F.create_map(_label(sector, 0), _time_to_spend_text(F.col("e.avg_range_min"), sector)))
         .alias("time_to_spend"),  # COL-003 — D-16
        F.to_json(F.create_map(_label(sector, 1), _best_time_text(F.col("e.canonical_candidate"))))
         .alias("best_time"),  # D-05
        cost.alias("cost"),  # COL-007 — D-08
        _cost_vnd(cost, sector).alias("cost_vnd"),  # D-15
        F.coalesce(F.col("am.service_flag"), F.lit("[]")).alias("service_flag"),  # D-12
        # END_INFO_SELECT
    )


SPEC = NodeSpec(
    trg_tbl="gld_enr_poi_info",
    inputs={
        "entity": "slv_poi_entity_non_enrich", "amenity": "slv_poi_amenity",
        "rating": "slv_3p_poi_rating", "price": "slv_3p_poi_price", "hours": "slv_3p_poi_opening_hours",
        "review": "slv_3p_poi_review", "i18n": "slv_pn_business_service_i18ns",
    },
    pk=("poi_id",),
    build=build,
    required_inputs=("entity",),
    block_checks=(
        ("json_text_null", "time_to_spend IS NULL OR best_time IS NULL OR cost_vnd IS NULL OR service_flag IS NULL"),
        ("cost_vnd_mismatch", "(cost IS NULL) <> (cost_vnd = 'null')"),
        ("google_cols_not_google", "coalesce(lower(trim(source_name)), '') <> 'google' AND (rating IS NOT NULL "
                                   "OR user_rating_count IS NOT NULL OR price_level IS NOT NULL OR review_json IS NOT NULL)"),
        ("opening_hours_other_source", "coalesce(lower(trim(source_name)), '') NOT IN ('google', 'partner') "
                                       "AND opening_hours_text IS NOT NULL"),
    ),
    warn_checks=(
        ("cost_null", "cost IS NULL"),
        ("time_to_spend_unfixed", "avg_range_min IS NULL OR avg_range_min <= 0"),
        ("partner_opening_hours_null", "lower(trim(source_name)) = 'partner' AND opening_hours_text IS NULL"),
    ),
    write_mode="APPEND",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
